# Day 16: Advanced Pandas Techniques
Demonstrating pivot_table, melt, multi-index, merge, apply/lambda on sales data.

In [1]:
import pandas as pd
df = pd.read_csv('../Data/Processed/cleaned_sales.csv')
df.head()

,order_id,order_date,customer_name,city,region,salesperson,category,product,quantity,unit_price,discount_pct,total_amount,payment_mode,status
0,ORD-1116,2023-10-22,Customer_687,Mumbai,West,Karan Singh,Electronics,Headphones,8.0,5699.16,0.0,45593.28,Net Banking,Returned
1,ORD-1121,2023-08-29,Customer_746,Ahmedabad,west,Neha Joshi,Sports,Cycle,6.0,14909.64,5.0,84984.95,UPI,Delivered
2,ORD-1123,2023-01-22,Customer_929,Lucknow,North,Neha Joshi,Sports,Cycle,8.0,15081.47,20.0,96521.41,Card,Delivered
3,ORD-1227,2023-10-13,Customer_556,Jaipur,North,Unassigned,Electronics,Laptop,4.0,66604.54,20.0,213134.53,UPI,Returned
4,ORD-1067,2023-03-22,Customer_277,Chandigarh,North,Sneha Patel,Sports,Cricket Bat,4.0,2235.29,10.0,8047.04,upi,Delivered


## 1. Pivot Table
Summarize total revenue by region and category.

In [2]:
pivot_df = df.pivot_table(values='total_amount', index='region', columns='category', aggfunc='sum', fill_value=0)
pivot_df

category,Clothing,Cloting,Electronic,Electronics,Electronis,Furniture,Furnitures,Furnture,Office Supplies,Sport,Sports,Sprots
region,,,,,,,,,,,,
East,7907501.51,0.00,0.00,1296807.30,0.00,986548.26,0.00,216267.12,45336.11,0.00,186243.22,3700.35
North,200356.83,14986.18,26647.30,1227693.17,0.00,243424.70,51683.44,22752.81,81766.32,17894.92,264122.01,0.00
South,107723.22,0.00,202217.85,320641.00,0.00,362871.65,0.00,0.00,227395.02,0.00,85627950.16,0.00
West,13972475.45,0.00,0.00,1666846.92,181698.23,508917.41,0.00,0.00,157929.77,18072.88,486901.27,0.00
east,0.00,0.00,0.00,166846.66,0.00,0.00,0.00,0.00,22265.55,0.00,0.00,0.00
north,0.00,0.00,0.00,136820.39,0.00,21455.37,0.00,0.00,2266.11,0.00,0.00,0.00
south,8490.10,0.00,0.00,0.00,0.00,60414.62,0.00,0.00,0.00,0.00,4652.19,0.00
west,0.00,0.00,0.00,553192.62,0.00,96075.05,0.00,0.00,20202.55,0.00,84984.95,0.00


## 2. Melt
Unpivot the pivot table back to long format.

In [3]:
melted_df = pivot_df.reset_index().melt(id_vars=['region'], value_vars=pivot_df.columns, var_name='category', value_name='total_revenue')
melted_df.head()

,region,category,total_revenue
0,East,Clothing,7907501.51
1,North,Clothing,200356.83
2,South,Clothing,107723.22
3,West,Clothing,13972475.45
4,east,Clothing,0.00


## 3. Multi-Index
Create a multi-index dataframe by grouping region and category.

In [4]:
multi_idx_df = df.groupby(['region', 'category'])['total_amount'].sum().to_frame()
multi_idx_df.head(10)

total_amount
region category                     
East   Clothing           7907501.51
       Electronics        1296807.30
       Furniture           986548.26
       Furnture            216267.12
       Office Supplies      45336.11
       Sports              186243.22
       Sprots                3700.35
North  Clothing            200356.83
       Cloting              14986.18
       Electronic           26647.30

## 4. Apply / Lambda
Categorize orders into 'High Value' or 'Low Value' based on AOV.

In [5]:
aov = df['total_amount'].mean()
df['order_value_tier'] = df['total_amount'].apply(lambda x: 'High' if x > aov else 'Low')
df['order_value_tier'].value_counts()

order_value_tier
Low     203
High      5
Name: count, dtype: int64

## 5. Merge
Merge with a mock customer table.

In [6]:
customers = pd.DataFrame({'customer_name': df['customer_name'].unique(), 'segment': 'Retail'})
merged_df = df.merge(customers, on='customer_name', how='left')
merged_df.head()

,order_id,order_date,customer_name,city,region,salesperson,category,product,quantity,unit_price,discount_pct,total_amount,payment_mode,status,order_value_tier,segment
0,ORD-1116,2023-10-22,Customer_687,Mumbai,West,Karan Singh,Electronics,Headphones,8.0,5699.16,0.0,45593.28,Net Banking,Returned,Low,Retail
1,ORD-1121,2023-08-29,Customer_746,Ahmedabad,west,Neha Joshi,Sports,Cycle,6.0,14909.64,5.0,84984.95,UPI,Delivered,Low,Retail
2,ORD-1123,2023-01-22,Customer_929,Lucknow,North,Neha Joshi,Sports,Cycle,8.0,15081.47,20.0,96521.41,Card,Delivered,Low,Retail
3,ORD-1227,2023-10-13,Customer_556,Jaipur,North,Unassigned,Electronics,Laptop,4.0,66604.54,20.0,213134.53,UPI,Returned,Low,Retail
4,ORD-1067,2023-03-22,Customer_277,Chandigarh,North,Sneha Patel,Sports,Cricket Bat,4.0,2235.29,10.0,8047.04,upi,Delivered,Low,Retail
